# 🚀 ProfitLeak AI - Dual-Detector Machine Learning Pipeline (v6)
**Projek:** Deteksi Transaksi Tidak Ekonomis & Kebocoran Biaya Pengiriman E-Commerce  
**Penulis:** Galang (ML / Data Engineer) & Irsyad (Backend & ML Engineer)  
**Versi Model:** `v6` (Dual-Detector: Tarif Ongkir + Risiko Margin)  
**Artefak:** `model/profitleak_final_model_v5.pkl` & `model/profitleak_final_model_info_v5.json`

---

### 📌 Latar Belakang & Masalah Bisnis
Di industri e-commerce, merchant sering kali mengalami kebocoran keuntungan (*profit leak*) dari biaya logistik karena dua penyebab utama:
1. **Overcharging Kurir**: Ekspedisi menagih ongkos kirim jauh di atas tarif wajar untuk rute dan spesifikasi paket tertentu.
2. **Transaksi Tidak Ekonomis**: Barang bernilai murah dikirimkan ke lokasi terpencil atau berukuran bulky, sehingga ongkos kirim menyerap porsi sangat besar (>50%) dari harga jual barang, menghabiskan seluruh laba merchant.

ProfitLeak AI v6 mengkombinasikan **dua detektor cerdas** untuk menangkap kedua skenario tersebut secara presisi.

## 1. Import Library & Setup Lingkungan

In [ ]:
import io
import json
import os
import sys
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.model_selection import train_test_split

print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print(f"Joblib : {joblib.__version__}")

## 2. Definisi Parameter Model & Formula Jarak (Haversine)
Jarak pengiriman dihitung secara akurat menggunakan formula *Great-Circle Distance* (Haversine) antara koordinat penjual dan pembeli.

In [ ]:
RANDOM_STATE = 42
CURRENCY = "BRL"

# 5 Fitur Aman (Varian A)
BASE_FEATURES = ["weight_kg", "distance_km", "volume_cm3", "price_log"]
SAFE_EXTRA = ["density_g_cm3"]
FEATURES_V6 = BASE_FEATURES + SAFE_EXTRA

def haversine_km(lat1, lon1, lat2, lon2):
    """Menghitung jarak geolokasi dalam kilometer."""
    r = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))

def r2_score_manual(y_true, y_pred):
    """Koefisien determinasi R2 score."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = ((y_true - y_pred) ** 2).sum()
    ss_tot = ((y_true - y_true.mean()) ** 2).sum()
    return float(1 - ss_res / ss_tot)

print("Fitur input model v6:", FEATURES_V6)

## 3. Verifikasi Artefak Model v6
Kita memuat bundle model `model/profitleak_final_model_v5.pkl` dan metadata `model/profitleak_final_model_info_v5.json` untuk mengaudit seluruh komponennya.

In [ ]:
model_path = Path("model/profitleak_final_model_v5.pkl")
info_path = Path("model/profitleak_final_model_info_v5.json")

assert model_path.exists(), f"Model tidak ditemukan di {model_path}"
assert info_path.exists(), f"Info model tidak ditemukan di {info_path}"

bundle = joblib.load(model_path)
with open(info_path, "r", encoding="utf-8") as f:
    info = json.load(f)

print(f"Versi Model           : {bundle.get('version')}")
print(f"Detektor 1 (Rate)     : {type(bundle.get('rate_model')).__name__}")
print(f"Detektor 2 (Margin)   : {type(bundle.get('margin_risk_model')).__name__}")
print(f"Threshold Rasio Tarif : {bundle.get('threshold'):.5f}")
print(f"Threshold Prob Margin : {bundle.get('margin_proba_threshold')}")
print(f"R2 Test Score         : {info.get('r2_test')}")
print(f"F1 Score Gabungan     : {info.get('evaluation_test', {}).get('combined', {}).get('f1')}")

## 4. Analisis Eksperimen: Pencegahan Target Leakage (Varian A vs Varian B)

> 🛡️ **Pentingnya Integritas Data dalam Hackathon:**  
> Saat eksplorasi fitur, tim menguji dua varian:
> - **Varian A (Aman)**: Hanya menggunakan fitur paket & harga (`weight_kg`, `distance_km`, `volume_cm3`, `price_log`, `density_g_cm3`).
> - **Varian B (Leaky)**: Menambahkan fitur turunan yang melibatkan `freight_value` (`freight_ratio`, `freight_per_km_kg`).
>
> **Hasil**: Pada Varian B, $R^2$ melonjak drastis dari **0.7349** menjadi **0.9955**. Namun, ini adalah **Target Leakage palsu** karena sistem inferensi tidak boleh membocorkan target sebelum prediksi dibuat. Oleh karena itu, **Varian B ditolak secara tegas**, dan Varian A yang jujur dipilih sebagai model produksi.

In [ ]:
variant_compare = info.get("r2_variant_compare", {})
print("Perbandingan R2 Varian Fitur:")
for variant, score in variant_compare.items():
    print(f"  - {variant:25s}: {score}")

print("\nKeputusan Ilmiah:", info.get("feature_variant_decision"))

## 5. Mengapa Diperlukan Detektor Margin? (Evaluasi Dua Detektor)
Pada data uji nyata ($N = 16.754$), ground truth transaksi merugi didefinisikan sebagai transaksi di mana ongkir menyerap $>50\%$ dari harga produk.

- **Detektor Tarif (`rate_model`)**: Sangat akurat untuk mark-up liar kurir ($>3.09\times$), tetapi memiliki Recall rendah ($0.63\%$) jika digunakan sendirian untuk transaksi rugi karena rata-rata ongkir transaksi rugi sebenarnya bertarif normal.
- **Detektor Margin (`margin_risk_model`)**: Berhasil mendeteksi risiko margin dengan Precision $88.94\%$ dan Recall $83.14\%$.
- **Sistem Gabungan**: Menghasilkan F1-score **$86.09\%$** dengan Precision **$88.70\%$**.

In [ ]:
eval_data = info.get("evaluation_test", {})
df_eval = pd.DataFrame({
    "Detektor Tarif (Rate)": eval_data.get("rate", {}),
    "Detektor Margin (Margin)": eval_data.get("margin", {}),
    "Sistem Gabungan (Combined)": eval_data.get("combined", {})
}).T[["tp", "fp", "fn", "tn", "precision", "recall", "f1"]]

display(df_eval)

### 📊 Visualisasi Performa: Confusion Matrix & Skor F1

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Bar Chart Perbandingan Precision, Recall, F1
labels = ['Rate Detector', 'Margin Detector', 'Combined Dual']
x = np.arange(len(labels))
width = 0.25

p_scores = [eval_data['rate']['precision'], eval_data['margin']['precision'], eval_data['combined']['precision']]
r_scores = [eval_data['rate']['recall'], eval_data['margin']['recall'], eval_data['combined']['recall']]
f1_scores = [eval_data['rate']['f1'], eval_data['margin']['f1'], eval_data['combined']['f1']]

ax1.bar(x - width, p_scores, width, label='Precision', color='#3b82f6')
ax1.bar(x, r_scores, width, label='Recall', color='#f59e0b')
ax1.bar(x + width, f1_scores, width, label='F1-Score', color='#10b981')

ax1.set_ylabel('Skor (0.0 - 1.0)')
ax1.set_title('Perbandingan Detektor pada Data Uji Nyata (N=16.754)')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylim(0, 1.1)
ax1.legend(loc='upper left')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# 2. Confusion Matrix Detektor Margin
cm = np.array([
    [eval_data['margin']['tn'], eval_data['margin']['fp']],
    [eval_data['margin']['fn'], eval_data['margin']['tp']]
])
ax2.imshow(cm, cmap='Blues')
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
ax2.set_xticklabels(['Prediksi Sehat', 'Prediksi Anomali'])
ax2.set_yticklabels(['Aktual Sehat', 'Aktual Anomali'])
ax2.set_title('Confusion Matrix Detektor Margin')

for i in range(2):
    for j in range(2):
        ax2.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                 color='white' if cm[i, j] > cm.max()/2 else 'black',
                 fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

## 6. Simulasi Inferensi Interaktif pada Berbagai Skenario Transaksi
Kita menguji 3 skenario nyata:
1. **Paket Normal**: Pengiriman sehat, tarif wajar, margin aman.
2. **Overcharging Kurir**: Ekspedisi menagih tarif $8\times$ di atas tarif wajar.
3. **Margin Tipis**: Barang murah (18 BRL) dikirim jarak jauh dengan ongkir 22 BRL ($122\%$ dari harga jual).

In [ ]:
rate_model = bundle["rate_model"]
risk_model = bundle["margin_risk_model"]
threshold_ratio = bundle["threshold"]
threshold_proba = bundle["margin_proba_threshold"]

def analisa_transaksi(weight_kg, distance_km, volume_cm3, price, actual_rate):
    density = (weight_kg * 1000.0) / max(volume_cm3, 100.0)
    price_log = np.log1p(price)
    
    # Input untuk rate_model
    feat_rate = pd.DataFrame([{
        "weight_kg": weight_kg,
        "distance_km": max(distance_km, 0.1),
        "volume_cm3": volume_cm3,
        "price_log": price_log,
        "density_g_cm3": density
    }])[bundle["rate_features"]]
    
    pred_log = rate_model.predict(feat_rate)[0]
    predicted_rate = float(np.expm1(pred_log))
    ratio = actual_rate / predicted_rate
    is_rate_leak = ratio > threshold_ratio
    
    # Input untuk margin_risk_model
    feat_risk = pd.DataFrame([{
        "weight_kg": weight_kg,
        "distance_km": max(distance_km, 0.1),
        "volume_cm3": volume_cm3,
        "density_g_cm3": density,
        "price_log": price_log
    }])[bundle["margin_risk_features"]]
    
    proba_risk = float(risk_model.predict_proba(feat_risk)[0][1])
    is_margin_risk = proba_risk >= threshold_proba
    
    is_leak = is_rate_leak or is_margin_risk
    
    return {
        "Ongkir Aktual": f"{actual_rate:.2f} BRL",
        "Tarif Wajar": f"{predicted_rate:.2f} BRL",
        "Rasio Tarif": f"{ratio:.2f}x (Ambang {threshold_ratio:.2f}x)",
        "Rate Leak?": "🚨 YA" if is_rate_leak else "✅ AMAN",
        "Prob Margin Risk": f"{proba_risk*100:.1f}% (Ambang {threshold_proba*100:.0f}%)",
        "Margin Risk?": "🚨 YA" if is_margin_risk else "✅ AMAN",
        "Status Akhir": "🚨 LEAK (BOCOR)" if is_leak else "✅ NORMAL (SEHAT)"
    }

skenario_uji = {
    "1. Normal Standard": (0.8, 120.0, 2000.0, 150.0, 16.0),
    "2. Ongkir Mahal (Kurir Liar)": (0.5, 50.0, 1000.0, 180.0, 75.0),
    "3. Margin Tipis (Harga Murah)": (2.5, 800.0, 12000.0, 18.0, 22.0)
}

hasil_simulasi = [dict(Skenario=nama, **analisa_transaksi(*params)) for nama, params in skenario_uji.items()]
display(pd.DataFrame(hasil_simulasi))

## 7. Batasan Cakupan Model & Kejujuran Data (*Honest Scope*)

Saat presentasi di depan juri, kami secara transparan menyatakan batasan model:
1. **Cakupan Model**:
   - Mengidentifikasi tarif ekspedisi di atas batas toleransi wajar (overcharging).
   - Mengidentifikasi pesanan di mana ongkir menyerap porsi berlebih dari harga jual barang.
2. **Batasan Data**:
   - Dataset publik e-commerce Olist **tidak mencantumkan harga modal (COGS/HPP)** dan **kolom diskon**.
   - Model kami tidak membuat asumsi fiktif mengenai modal barang; deteksi didasarkan murni pada rasio harga jual aktual terhadap struktur ongkir pengiriman.